# FD002 - comparacion con FD001/FD003 y busqueda LGBM/XGB

Este notebook documenta la decision final para FD002 despues de instalar `lightgbm` y `xgboost`.

La pregunta central es si conviene usar el mismo modelo que funciono para FD001 y FD003 o adaptar el enfoque para las seis condiciones operativas de FD002.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "CMAPSSData").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results" / "FD002"
CONFIGS_DIR = PROJECT_ROOT / "configs" / "FD002"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
FIGURES_DIR = PROJECT_ROOT / "figures" / "FD002"

pd.set_option("display.max_columns", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 1. Referencia: que paso en FD001 y FD003

FD001 y FD003 terminaron convergiendo a una misma familia: LightGBM temporal con `window_size=50`, `rul_cap=125`, objetivo `quantile`, `alpha=0.4`.

La diferencia es que FD003 necesito features adicionales sensibles a patrones de falla (`fault_sensitive`). Esa es una pista importante: se conserva el protocolo y la familia, pero se adapta la representacion cuando cambia el problema.

In [2]:
reference_rows = []
for label, config_path in [
    ("FD001", PROJECT_ROOT / "configs" / "FD001" / "fd001_final_quantile_candidate_notebook18_config.json"),
    ("FD003", PROJECT_ROOT / "configs" / "FD003" / "fd003_final_candidate_config.json"),
]:
    with config_path.open("r", encoding="utf-8") as file:
        config = json.load(file)
    metrics = config.get("validation_summary") or config.get("global_comparison_metrics", {})
    reference_rows.append({
        "subset": label,
        "model_family": config.get("model_family"),
        "model_name": config.get("model_name"),
        "feature_set": config.get("feature_set") or config.get("approach"),
        "window_size": config.get("window_size"),
        "rul_cap": config.get("rul_cap"),
        "objective": config.get("objective"),
        "alpha": config.get("alpha"),
        "sample_weight_scheme": config.get("sample_weight_scheme"),
        "mean_rmse": metrics.get("mean_rmse") or metrics.get("mean_RMSE"),
        "mean_cmapss": metrics.get("mean_cmapss_score") or metrics.get("mean_CMAPSS"),
        "mean_dangerous_error_pct": metrics.get("mean_dangerous_error_pct"),
    })

pd.DataFrame(reference_rows)

,subset,model_family,model_name,feature_set,window_size,rul_cap,objective,alpha,sample_weight_scheme,mean_rmse,mean_cmapss,mean_dangerous_error_pct
0,FD001,LightGBM,candidate_03_B_quantile_a040_search_14,None,50,125,quantile,0.4000,none,15.0880,307.4446,5.0447
1,FD003,LightGBM,fd003_lgbm_w50_cap125_quantile_a04_none_fault_...,fault_sensitive,50,125,quantile,0.4000,none,16.3670,404.1823,5.0000


## 2. Diferencia tecnica de FD002

FD002 tiene seis condiciones operativas y un unico modo de falla. Las conditions cambian dentro de cada motor, por lo que una ventana temporal mezcla dos efectos: regimen operativo y degradacion.

Por eso se probaron dos ideas:

- transferencia directa del modelo LightGBM de FD001/FD003;
- transferencia adaptada: mismo protocolo temporal, pero con `condition_id`, one-hot de condition y sensores normalizados por condition.

## 3. Comparacion LGBM/XGB

Con `lightgbm` y `xgboost` disponibles, se compararon candidatos directos: receta LGBM FD001/FD003 cruda, la misma receta con features condition-normalized, variantes LGBM con pesos y XGBoost con/sin pesos.

In [3]:
external_comparison = pd.read_csv(RESULTS_DIR / "fd002_lgbm_xgb_model_comparison.csv")
external_comparison[[
    "candidate_label",
    "model_type",
    "feature_set",
    "sample_weight_scheme",
    "mae",
    "rmse",
    "cmapss_score",
    "dangerous_error_pct",
]].head(10)

,candidate_label,model_type,feature_set,sample_weight_scheme,mae,rmse,cmapss_score,dangerous_error_pct
0,xgb_squarederror_condition_normalized_weighted,xgboost,condition_normalized,bin_weights,12.7212,16.1543,"1,036.8348",10.4651
1,lgbm_quantile_a04_condition_normalized_weighted,lightgbm,condition_normalized,bin_weights,12.7550,16.4190,"1,043.2055",6.2016
2,fd001_fd003_lgbm_reference_condition_normalized,lightgbm,condition_normalized,none,13.0529,16.4222,"1,085.1721",11.2403
3,lgbm_regression_condition_normalized_weighted,lightgbm,condition_normalized,bin_weights,13.6200,17.0404,"1,216.1736",12.4031
4,xgb_squarederror_condition_normalized,xgboost,condition_normalized,none,13.4975,16.9898,"1,261.7098",12.7907
5,fd001_fd003_lgbm_reference_raw,lightgbm,raw,none,18.7687,23.8704,"4,561.0136",19.3798


## 4. Busqueda de hiperparametros

Despues se hizo una busqueda acotada sobre LightGBM y XGBoost usando la representacion `condition_normalized`. La busqueda varia ventana temporal, regularizacion, profundidad/cantidad de hojas, learning rate, cantidad de estimadores y esquema de pesos.

In [4]:
search = pd.read_csv(RESULTS_DIR / "fd002_lgbm_xgb_hyperparam_search.csv")
search[[
    "candidate_label",
    "model_type",
    "window_size",
    "sample_weight_scheme",
    "mae",
    "rmse",
    "cmapss_score",
    "dangerous_error_pct",
    "params",
]].head(14)

,candidate_label,model_type,window_size,sample_weight_scheme,mae,rmse,cmapss_score,dangerous_error_pct,params
0,lgbm_search_09_quantile_w70,lightgbm,70,none,12.7271,16.4024,"1,093.5167",8.9147,"{""alpha"":0.35,""colsample_bytree"":0.9,""learning..."
1,xgb_search_03_squarederror_w50,xgboost,50,bin_weights,12.8746,16.5579,"1,097.5886",10.4651,"{""colsample_bytree"":0.8,""learning_rate"":0.03,""..."
2,xgb_search_01_squarederror_w50,xgboost,50,bin_weights,13.3576,16.7276,"1,132.1197",12.0155,"{""colsample_bytree"":0.9,""learning_rate"":0.05,""..."
3,lgbm_search_03_quantile_w50,lightgbm,50,soft,12.8930,16.5757,"1,178.5123",10.8527,"{""alpha"":0.45,""colsample_bytree"":0.8,""learning..."
4,lgbm_search_05_regression_w50,lightgbm,50,bin_weights,13.4227,16.8998,"1,180.5311",10.8527,"{""colsample_bytree"":1.0,""learning_rate"":0.05,""..."
5,lgbm_search_01_regression_w70,lightgbm,70,none,13.3535,16.7018,"1,194.0659",11.6279,"{""colsample_bytree"":0.8,""learning_rate"":0.05,""..."
6,lgbm_search_06_regression_w70,lightgbm,70,soft,12.9009,16.3095,"1,200.0975",11.2403,"{""colsample_bytree"":0.8,""learning_rate"":0.03,""..."
7,lgbm_search_04_regression_w50,lightgbm,50,soft,13.7997,17.3189,"1,267.6875",12.0155,"{""colsample_bytree"":0.9,""learning_rate"":0.05,""..."
8,xgb_search_00_squarederror_w50,xgboost,50,none,13.5771,17.1840,"1,304.9782",12.4031,"{""colsample_bytree"":0.8,""learning_rate"":0.03,""..."
9,xgb_search_02_squarederror_w50,xgboost,50,none,13.7209,17.3054,"1,352.8299",14.7287,"{""colsample_bytree"":0.9,""learning_rate"":0.05,""..."


## 5. Ranking final

El ranking final une los candidatos HGB previos, la comparacion LGBM/XGB y la busqueda LGBM/XGB. La seleccion sigue el criterio usado en el proyecto: C-MAPSS primero, RMSE y dangerous error como criterios secundarios.

In [5]:
ranking = pd.read_csv(RESULTS_DIR / "fd002_final_candidate_ranking.csv")
ranking[[
    "source",
    "candidate_label",
    "model_type",
    "feature_set",
    "window_size",
    "sample_weight_scheme",
    "mae",
    "rmse",
    "cmapss_score",
    "dangerous_error_pct",
]].head(12)

,source,candidate_label,model_type,feature_set,window_size,sample_weight_scheme,mae,rmse,cmapss_score,dangerous_error_pct
0,lgbm_xgb_comparison,xgb_squarederror_condition_normalized_weighted,xgboost,condition_normalized,50,bin_weights,12.7212,16.1543,"1,036.8348",10.4651
1,lgbm_xgb_comparison,lgbm_quantile_a04_condition_normalized_weighted,lightgbm,condition_normalized,50,bin_weights,12.7550,16.4190,"1,043.2055",6.2016
2,lgbm_xgb_comparison,fd001_fd003_lgbm_reference_condition_normalized,lightgbm,condition_normalized,50,none,13.0529,16.4222,"1,085.1721",11.2403
3,lgbm_xgb_search,lgbm_search_09_quantile_w70,lightgbm,condition_normalized,70,none,12.7271,16.4024,"1,093.5167",8.9147
4,lgbm_xgb_search,xgb_search_03_squarederror_w50,xgboost,condition_normalized,50,bin_weights,12.8746,16.5579,"1,097.5886",10.4651
5,lgbm_xgb_search,xgb_search_01_squarederror_w50,xgboost,condition_normalized,50,bin_weights,13.3576,16.7276,"1,132.1197",12.0155
6,sklearn_hgb_search,hgb_search_03_absolute_error_w50,hist_gradient_boosting,condition_normalized,50,bin_weights,12.6942,16.1871,"1,137.5722",10.0775
7,lgbm_xgb_search,lgbm_search_03_quantile_w50,lightgbm,condition_normalized,50,soft,12.8930,16.5757,"1,178.5123",10.8527
8,lgbm_xgb_search,lgbm_search_05_regression_w50,lightgbm,condition_normalized,50,bin_weights,13.4227,16.8998,"1,180.5311",10.8527
9,lgbm_xgb_search,lgbm_search_01_regression_w70,lightgbm,condition_normalized,70,none,13.3535,16.7018,"1,194.0659",11.6279


## 6. Modelo final

El modelo final no es exactamente el mismo que FD001/FD003.

FD001 y FD003 usan LightGBM quantile. Para FD002, el mejor candidato por C-MAPSS fue XGBoost `reg:squarederror` con features `condition_normalized` y pesos por bins de RUL.

La conclusion importante no es que LightGBM falle: de hecho, el LightGBM transferido con condition-normalized queda muy competitivo. Pero FD002 se beneficia de cambiar tanto la representacion como la familia final.

In [6]:
validation_metrics = pd.read_csv(RESULTS_DIR / "fd002_best_validation_metrics.csv")
official_metrics = pd.read_csv(RESULTS_DIR / "fd002_official_test_metrics.csv")
with (CONFIGS_DIR / "fd002_best_model_config.json").open("r", encoding="utf-8") as file:
    best_config = json.load(file)

summary = pd.concat(
    [
        validation_metrics.assign(split="artificial_validation"),
        official_metrics.assign(split="official_test"),
    ],
    ignore_index=True,
    sort=False,
)
summary[[
    "split",
    "model_name",
    "representation",
    "mae",
    "rmse",
    "r2",
    "cmapss_score",
    "cmapss_score_mean",
    "dangerous_error_pct",
]]

,split,model_name,representation,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct
0,artificial_validation,xgb_squarederror_condition_normalized_weighted,temporal_w50_condition_normalized,12.7212,16.1543,0.8539,"1,036.8348",4.0187,10.4651
1,official_test,xgb_squarederror_condition_normalized_weighted,temporal_w50_condition_normalized,16.6423,25.0253,0.7835,"4,893.5633",18.8941,7.3359


In [7]:
best_config["final_model"]

{'candidate_label': 'xgb_squarederror_condition_normalized_weighted',
 'model_type': 'xgboost',
 'representation': 'temporal_w50_condition_normalized',
 'sample_weight_scheme': 'bin_weights',
 'hyperparameters': {'colsample_bytree': 0.85,
  'learning_rate': 0.04,
  'max_depth': 3,
  'n_estimators': 650,
  'objective': 'reg:squarederror',
  'reg_alpha': 0.0,
  'reg_lambda': 5.0,
  'subsample': 0.85}}

In [8]:
best_config["comparison_with_fd001_fd003"]

{'fd001_final': 'LightGBM temporal_w50, RUL cap 125, quantile alpha=0.4, no sample weights.',
 'fd003_final': 'LightGBM temporal_w50, RUL cap 125, quantile alpha=0.4, fault-sensitive features, no sample weights.',
 'fd002_final': 'XGBoost temporal_w50, RUL cap 125, condition-normalized features, squared-error objective, bin_weights.',
 'same_model_as_fd001_fd003': False,
 'reason': 'FD002 needs explicit operating-condition handling; XGBoost weighted gave the lowest artificial-validation C-MAPSS among tested LGBM/XGB candidates.'}

## 7. Artifacts

El test oficial no se uso para elegir hiperparametros; solo se reporta despues de fijar el candidato final.

In [9]:
artifacts = {
    "config": CONFIGS_DIR / "fd002_best_model_config.json",
    "final_ranking": RESULTS_DIR / "fd002_final_candidate_ranking.csv",
    "lgbm_xgb_comparison": RESULTS_DIR / "fd002_lgbm_xgb_model_comparison.csv",
    "lgbm_xgb_search": RESULTS_DIR / "fd002_lgbm_xgb_hyperparam_search.csv",
    "validation_metrics": RESULTS_DIR / "fd002_best_validation_metrics.csv",
    "validation_predictions": RESULTS_DIR / "fd002_best_validation_predictions.csv",
    "official_predictions": PREDICTIONS_DIR / "fd002_best_model_predictions.csv",
    "official_metrics": RESULTS_DIR / "fd002_official_test_metrics.csv",
    "figures": FIGURES_DIR,
}
pd.DataFrame([{"artifact": key, "path": str(value.relative_to(PROJECT_ROOT))} for key, value in artifacts.items()])

,artifact,path
0,config,configs\FD002\fd002_best_model_config.json
1,final_ranking,results\FD002\fd002_final_candidate_ranking.csv
2,lgbm_xgb_comparison,results\FD002\fd002_lgbm_xgb_model_comparison.csv
3,lgbm_xgb_search,results\FD002\fd002_lgbm_xgb_hyperparam_search...
4,validation_metrics,results\FD002\fd002_best_validation_metrics.csv
5,validation_predictions,results\FD002\fd002_best_validation_prediction...
6,official_predictions,predictions\fd002_best_model_predictions.csv
7,official_metrics,results\FD002\fd002_official_test_metrics.csv
8,figures,figures\FD002
